# Working code

LangGraph + LCEL version of LLM + dynamic tool chaining. Next step would be to incorporate vector db and custom deep seek

In [ ]:
# LangGraph + LCEL version of LLM + tool chaining
from langchain_openai import OpenAI
from langchain.memory import ConversationBufferMemory
from langchain.tools import Tool
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableSequence
from langgraph.graph import END, StateGraph
import re, os, json
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, ValidationError, field_validator
from typing import TypedDict, Optional

# Load environment variables
load_dotenv(find_dotenv())
api_key = os.environ['OPENAI_API_KEY']

# --- Data context ---
data_context = {"A": 3, "B": 12, "C": 23, "D": 7}

# --- Shared state ---
last_expr = {"expr": ""}
used_describe = {"done": False}
tool_call_log = {"get_data_context": 0, "describe_expression": 0}
data_context_fetched = {"done": False}

# --- Tools ---
def substitute_variables(expr: str) -> str:
    for var, val in data_context.items():
        expr = re.sub(rf"\b{re.escape(var)}\b", str(val), expr)
    print("Substituted expr = ", expr)
    return expr

def rewrite_expression(input_str: str) -> str:
    match = re.match(r"(?i)subtract (\d+) from (\d+)", input_str)
    return f"{match.group(2)} - {match.group(1)}" if match else input_str

def evaluate_expression(input_str: str) -> str:
    input_str = rewrite_expression(input_str)
    substituted_expr = substitute_variables(input_str.strip())
    last_expr["expr"] = substituted_expr
    print("Evaluating expression:", substituted_expr)
    try:
        if not re.fullmatch(r"[0-9+\-*/(). ]+", substituted_expr):
            return "Invalid characters in expression."
        return str(eval(substituted_expr))
    except Exception as e:
        return f"Invalid math expression: {e}"

def calculate_expression(input_str: str) -> str:
    print("inside calculate_expression tool, input_str = ", input_str)
    return evaluate_expression(input_str)

def get_data_context(input_str: str) -> str:
    tool_call_log["get_data_context"] += 1
    print("inside get_data_context tool, input_str = ", input_str)
    if data_context_fetched["done"]:
        return "(Reminder) You already have the variable values: " + ", ".join([f"{k} = {v}" for k, v in data_context.items()])
    try:
        data = ", ".join([f"{k} = {v}" for k, v in data_context.items()])
        data_context_fetched["done"] = True
        print("data = ", data)
        return data
    except Exception as e:
        return f"Error retrieving context: {e}"

def describe_expression(input_str: str) -> str:
    if used_describe["done"]:
        return "Expression already described."
    expr = last_expr["expr"]
    if not expr:
        return "No expression has been evaluated yet."
    try:
        result = str(eval(expr))
        used_describe["done"] = True
        tool_call_log["describe_expression"] += 1
        return f"This operation evaluated the expression: {expr}. The final result is {result}."
    except Exception as e:
        return f"Unable to evaluate the final expression: {e}"

def getdiscount(prodname: str, totalmount: int) -> int:
    pricingrules = [
        {"id": "1", "name": "Rule 1", "productname": "AWS", "maxtotalmount": 1000, "mintotalmount": 0, "discount": 3},
        {"id": "2", "name": "Rule 2", "productname": "AWS", "maxtotalmount": 10000, "mintotalmount": 1001, "discount": 5},
        {"id": "3", "name": "Rule 3", "productname": "GCP", "maxtotalmount": 1000, "mintotalmount": 0, "discount": 2},
        {"id": "4", "name": "Rule 4", "productname": "GCP", "maxtotalmount": 10000, "mintotalmount": 1001, "discount": 3},
        {"id": "5", "name": "Rule 5", "productname": "AWS", "maxtotalmount": 100000, "mintotalmount": 10001, "discount": 7},
        {"id": "6", "name": "Rule 6", "productname": "GCP", "maxtotalmount": 100000, "mintotalmount": 10001, "discount": 8}
    ]
    discount = 0
    print("inside getdiscount tool, input_str = ", prodname, "|", totalmount)
    for rule in pricingrules:
        if rule["productname"] == prodname and totalmount >= rule["mintotalmount"] and totalmount <= rule["maxtotalmount"]:
            discount = rule["discount"]
            break
    return discount

# --- LangChain Tool wrappers ---
get_data_tool = Tool(name="get_data_context", func=get_data_context, description="Fetch current variables")
calc_expr_tool = Tool(name="calculate_expression", func=calculate_expression, description="Evaluate math expression")
describe_expr_tool = Tool(name="describe_expression", func=describe_expression, description="Explain result")
discount_tool = Tool(
    name="get_discount",
    func=lambda input: str(getdiscount(**json.loads(input))),
    description="Get discount based on product name and total amount. Input should be a JSON string with 'prodname' and 'totalmount'."
)

# --- LLM output schema ---
class LLMResponse(BaseModel):
    requires_context: bool
    expression: Optional[str] = None
    describe_required: bool
    discount_input: Optional[dict] = None

    @field_validator('describe_required', mode='before')
    @classmethod
    def coerce_bool(cls, v):
        if isinstance(v, bool): return v
        if isinstance(v, str): return v.lower() in ['true', 'yes', 'describe', 'discount']
        return bool(v)

    @field_validator("discount_input", mode="before")
    @classmethod
    def parse_discount_input(cls, v):
        if v is None:
            return None
        if isinstance(v, dict):
            return v
        if isinstance(v, str):
            try:
                parsed = json.loads(v)
                if isinstance(parsed, dict):
                    return parsed
                else:
                    raise ValueError("discount_input must be a dictionary")
            except Exception:
                raise ValueError("discount_input must be a dictionary or valid JSON string")
        raise ValueError("discount_input must be a dictionary")

# --- Graph state schema ---
class GraphState(TypedDict):
    question: str
    parsed: Optional[LLMResponse]
    context: Optional[str]
    result: Optional[str]
    description: Optional[str]
    error: Optional[str]
    response: Optional[str]

# --- LLM ---
llm = OpenAI(temperature=0, api_key=api_key)
memory = ConversationBufferMemory(return_messages=True)

# --- LangGraph Node Runnables ---
def llm_parser_node(state: GraphState):
    question = state["question"]
    prompt = f"""
    Analyze this user request and return in JSON with keys: requires_context, expression, describe_required, and optionally discount_input.

    Example:
    {{
        "requires_context": true,
        "expression": "A + B - C + D",
        "describe_required": true
    }}

    Or for discount:
    {{
        "requires_context": false,
        "expression": "",
        "describe_required": false,
        "discount_input": {{"prodname": "AWS", "totalmount": 12000}}
    }}

    User asked: {question}
    """
    raw = llm.invoke(prompt)
    print("\U0001F4AC LLM Raw Response:", raw)
    try:
        parsed = LLMResponse(**json.loads(raw))
        return {**state, "parsed": parsed}
    except Exception as e:
        return {**state, "parsed": None, "error": str(e)}

def get_context_node(state: GraphState):
    return {**state, "context": get_data_tool.run("")}

def calc_node(state: GraphState):
    parsed = state.get("parsed")
    expr = parsed.expression if parsed and parsed.expression else ""
    return {**state, "result": calc_expr_tool.run(expr)}

def describe_node(state: GraphState):
    return {**state, "description": describe_expr_tool.run("")}

def discount_node(state: GraphState):
    try:
        input_data = state["parsed"].discount_input if state.get("parsed") and state["parsed"].discount_input else None
        if input_data:
            result = discount_tool.run(json.dumps(input_data))
            return {**state, "result": f"Applicable discount: {result}%"}
        return {**state, "result": "No discount parameters provided."}
    except Exception as e:
        return {**state, "error": str(e)}

def final_output_node(state: GraphState):
    return {
        **state,
        "response": "\n".join(filter(None, [
            state.get("error"),
            state.get("context"),
            state.get("result"),
            state.get("description")
        ])) or "No response could be generated."
    }

# --- Graph ---
graph = StateGraph(GraphState)
graph.add_node("llm_parse", llm_parser_node)
graph.add_node("get_context", get_context_node)
graph.add_node("calculate", calc_node)
graph.add_node("describe", describe_node)
graph.add_node("discount", discount_node)
graph.add_node("output", final_output_node)

graph.set_entry_point("llm_parse")
graph.add_conditional_edges("llm_parse", lambda s: (
    "discount" if s.get("parsed") and s["parsed"].discount_input else
    "get_context" if s.get("parsed") and s["parsed"].requires_context else
    "calculate"
))
graph.add_edge("discount", "output")
graph.add_edge("get_context", "calculate")
graph.add_conditional_edges("calculate", lambda s: "describe" if s.get("parsed") and s["parsed"].describe_required else "output")
graph.add_edge("describe", "output")
graph.set_finish_point("output")
compiled_graph = graph.compile()

# --- Run test ---
used_describe["done"] = False
last_expr["expr"] = ""
tool_call_log = {"get_data_context": 0, "describe_expression": 0}
data_context_fetched["done"] = False

inputs = {"question": "What discount can I get for AWS with a total amount of 10000?"}
#inputs = {"question": "What values are in the data context? Then compute A + 2 * B - C + D and describe what you did"}
out = compiled_graph.invoke(inputs)
print("\n\U0001F3AF Final Answer:", out.get("response", "No response key found"))
